# M3 — Intégrer les mesures capteurs à la pipeline DiagOps

## Mission

L'exploitation ouvre l'export de sa supervision. Votre objectif est d'**intégrer cette source temporelle à la pipeline, la relier aux événements existants, faire évoluer les règles sans casser l'acquis de M2 et décider ce qui peut être transmis à M4**.

Ce notebook est une **trame d'investigation**, pas un pas à pas ni une correction. Vous pouvez modifier son organisation, déplacer du code dans des fonctions ou des scripts et ajouter les contrôles que vous jugez utiles. Chaque constat important doit être accompagné du résultat qui le soutient.

Rappel : les fichiers de `data_pack/` ne sont jamais modifiés. Toutes les sorties vont dans `output/`.

## 0. Environnement et point de départ

Les chemins ci-dessous recherchent la racine du dépôt S04 et utilisent son `data_pack/`. Ils fonctionnent depuis le starter de référence comme depuis `work/M3/`. La variable d'environnement `DIAGOPS_DATA_DIR` permet d'utiliser un autre emplacement sans modifier le notebook.

In [ ]:
from pathlib import Path
import hashlib
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPOSITORY_ROOT = next((root for root in roots if (root / 'data_pack' / 'MANIFEST.yaml').is_file()), None)
if os.environ.get('DIAGOPS_DATA_DIR'):
    DATA_DIR = Path(os.environ['DIAGOPS_DATA_DIR']).resolve()
elif REPOSITORY_ROOT is not None:
    DATA_DIR = REPOSITORY_ROOT / 'data_pack' / '2026-S1'
else:
    raise RuntimeError("Racine du dépôt introuvable : définissez DIAGOPS_DATA_DIR")

OUTPUT_DIR = Path('output')
for name in ('processed', 'aggregates', 'alignment'):
    (OUTPUT_DIR / name).mkdir(parents=True, exist_ok=True)

REFERENCE_DIR = DATA_DIR / 'reference_runs' / 'm2_for_m3'
print('données :', DATA_DIR)
print('référence M2 :', REFERENCE_DIR)

In [ ]:
SOURCE_FILES = {
    'equipment': DATA_DIR / 'equipment' / 'equipment.csv',
    'events': DATA_DIR / 'events' / 'events.csv',
    'maintenance': DATA_DIR / 'maintenance' / 'maintenance_history.csv',
    'sensors': DATA_DIR / 'sensors' / 'sensor_readings.csv',
}

missing_files = [str(path) for path in SOURCE_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Fichiers M3 introuvables : {missing_files}')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

sources = {name: pd.read_csv(path) for name, path in SOURCE_FILES.items()}
empreintes = pd.DataFrame(
    [
        {'source': name, 'lignes': len(sources[name]), 'octets': path.stat().st_size, 'sha256': sha256(path)}
        for name, path in SOURCE_FILES.items()
    ]
)
empreintes

**Trace initiale à conserver :** ces empreintes démontrent que le travail porte sur les fichiers reçus.

Notez l'écart de volumétrie entre les mesures et les trois tables de M2. Il change la manière de travailler : ce qui est acceptable sur 1 800 lignes ne l'est pas nécessairement sur 50 000.

## 0 bis. Choisir son point de départ M2

Deux options sont recevables : repartir de votre propre préparation M2, ou repartir de la référence commune. **Indiquez laquelle vous utilisez et pourquoi.**

La référence contient les tables préparées, la quarantaine, le registre des règles et la décision.

In [ ]:
reference = {
    'equipment': REFERENCE_DIR / 'processed' / 'equipment.csv',
    'events': REFERENCE_DIR / 'processed' / 'events.csv',
    'maintenance': REFERENCE_DIR / 'processed' / 'maintenance_history.csv',
    'quarantine': REFERENCE_DIR / 'quarantine' / 'quarantine_m2.csv',
}
reference_frames = {name: pd.read_csv(path) for name, path in reference.items()}
{name: len(frame) for name, frame in reference_frames.items()}

In [ ]:
# Registre des règles héritées, à reprendre avec un statut explicite.
print((REFERENCE_DIR / 'regles_m2.md').read_text(encoding='utf-8')[:1500])

## 1. Cadrer la source avant de la traiter

Avant tout contrôle, décrivez ce que vous avez reçu.

### Questions à traiter

- Que représente une ligne, et quelle clé identifie une mesure ?
- Quelles séries sont présentes : quels équipements, quels capteurs, quelles unités ?
- Quelle période est réellement couverte, comparée à la période annoncée dans `SCHEMA.md` ?
- Quel est le pas d'échantillonnage observé, et est-il régulier ?
- Le fichier déclare-t-il un identifiant de ligne ?
- Qui produit cette source, à quelle fréquence est-elle livrée, et que se passe-t-il si une livraison manque ?
- Que reste-t-il de disponible pour les équipements qu'elle ne couvre pas ?

In [ ]:
mesures = sources['sensors']
mesures.head()

In [ ]:
mesures.dtypes

In [ ]:
# Inventaire des séries : équipement, capteur, unité.
# À compléter : période couverte et pas observé par série.
mesures.groupby(['sensor_name', 'unit'], dropna=False).size().rename('mesures').reset_index()

Le starter fournit des primitives descriptives réutilisables ici :

```python
from src.data_pipeline.timeseries import to_utc, naive_timestamps, duplicated_keys, observed_steps, series_overview
```

Elles décrivent, elles ne décident pas. Le seuil au-delà duquel un écart d'échantillonnage devient un problème est votre décision.

In [ ]:
# TODO : période réellement couverte, pas médian et pas maximal par série.
# TODO : comparer au pas et à la période annoncés dans data_pack/SCHEMA.md.

## 2. Établir un diagnostic propre au temporel

Les contrôles de M2 portaient sur des lignes indépendantes. Une série demande des contrôles supplémentaires. Traitez au minimum :

- unicité de la clé logique, et distinction entre doublon strict et doublon porteur de deux valeurs ;
- homogénéité du format d'horodatage et présence d'un fuseau ;
- alignement des séries sur une grille commune ;
- continuité de l'échantillonnage et localisation des interruptions ;
- cohérence entre `sensor_name`, `unit` et ordre de grandeur ;
- plages physiques plausibles et valeurs sentinelles ;
- comportements de capteur : valeur figée, dérive lente, saut brutal ;
- équipement inconnu, période inattendue.

Chaque constat doit être **quantifié**, pas seulement cité.

In [ ]:
# TODO : unicité de la clé logique equipment_id + timestamp + sensor_name.
# Distinguez le doublon strict du doublon de clé à valeur divergente : ils ne
# se traitent pas de la même façon.

In [ ]:
# TODO : format des horodatages. Combien de valeurs sans fuseau ? Combien
# d'illisibles ? Quelle hypothèse retenez-vous, et que change-t-elle ?

In [ ]:
# TODO : cohérence entre nom de capteur, unité et ordre de grandeur des valeurs.

In [ ]:
# TODO : santé des capteurs. Une valeur constante sur une longue fenêtre, une
# dérive lente et un saut isolé ne se détectent pas avec le même contrôle.

### Visualiser avant de conclure

Un graphique par série suspecte aide à trancher entre défaut d'instrument et comportement d'équipement. Donnez un titre à chaque graphique et faites-le suivre d'une interprétation courte.

In [ ]:
# TODO : tracer une ou deux séries suspectes et les commenter.

## 3. Erreur, mesure réelle ou cas indécidable

Une valeur inhabituelle n'est pas nécessairement fausse. Pour **au moins trois observations atypiques de nature différente**, expliquez ce qui vous fait conclure et ce que vous décidez.

Le rapprochement avec `events.csv` et `maintenance_history.csv` fait partie des éléments disponibles : une élévation qui précède un incident enregistré ne se traite pas comme une valeur sentinelle isolée.

Une suppression silencieuse d'observation atypique est un défaut, pas une préparation.

In [ ]:
# TODO : trois cas argumentés, avec le résultat qui soutient chaque décision.

## 4. Registre de règles et non-régression

La pipeline porte maintenant deux familles de règles. Rendez-les visibles.

```python
from src.data_pipeline.rules import Rule, RuleRegistry
from contracts.schemas import M2_RULE_IDS
```

`RuleRegistry.missing(M2_RULE_IDS)` liste les règles héritées qui n'ont pas encore reçu de statut. Une règle `modifiee` ou `abandonnee` exige une justification.

In [ ]:
# TODO : déclarer le statut de chaque règle héritée, puis les règles ajoutées.
# TODO : exporter le registre dans output/registre_regles.csv et docs/registre_regles.md.

In [ ]:
# TODO : non-régression. Rejouez votre préparation M2 et comparez les sorties
# aux résultats de référence. Une différence doit être expliquée, pas découverte
# après coup.

## 5. Rapprocher les mesures et les événements

Une mesure et un événement ne partagent pas de clé : seuls l'équipement et le temps les rapprochent.

- choisissez une fenêtre — durée avant, durée après — et justifiez-la ;
- contrôlez la cardinalité : mesures par événement, événements sans mesure, mesures hors de toute fenêtre ;
- vérifiez que le rapprochement ne duplique pas les mesures sans le dire.

`window_bounds` construit les bornes ; le rapprochement lui-même est à écrire dans `contracts.schemas.align_measures_to_events`.

Deux approches répondent à deux questions différentes : `merge_asof` associe la mesure la plus proche, une jointure par intervalle associe toutes les mesures d'une fenêtre.

In [ ]:
# TODO : construire le rapprochement et contrôler sa cardinalité.
# TODO : écrire le résultat dans output/alignment/.

## 6. Agréger à un grain utilisable

Produisez au moins un jeu d'agrégats à un grain défini — par exemple équipement, capteur et fenêtre — avec des indicateurs simples : nombre de mesures, complétude, minimum, maximum, moyenne, écart-type.

Indiquez explicitement **ce que ce grain fait perdre**.

In [ ]:
# TODO : agrégats, puis écriture dans output/aggregates/.

## 7. Documenter le flux et le cycle de vie

Deux documents deviennent nécessaires dès qu'une source s'ajoute, et tous deux sont périssables : ils se mettent à jour, ils ne s'écrivent pas une fois.

**Le flux de traitement** : les fichiers d'entrée, les étapes traversées, ce qui est écrit à chacune, les points où une ligne peut être écartée, et la commande qui rejoue l'ensemble. Une autre personne doit pouvoir suivre une ligne d'un bout à l'autre.

**Le cycle de vie du jeu de données** : origine, fréquence de livraison, format et accès, durée et forme de conservation, qui accède aux données préparées, ce qui se passe à l'arrivée d'une nouvelle période, et à partir de quand ces données cessent d'être utilisables.

Le modèle `templates/flux_et_cycle_de_vie.md` en donne la trame. Nommez les destinataires : équipe technique, métier, ou personne chargée de la conformité.

In [ ]:
# TODO : chiffres à reprendre dans le flux — volumes entrants, volumes sortants,
# lignes écartées par étape. Le document se construit à partir de résultats, pas
# de souvenirs.

## 8. Couverture et risques

Le parc instrumenté n'est pas le parc complet. Mesurez la différence, puis examinez les risques introduits par cette source.

- combien d'équipements sont instrumentés, quels sites, types et criticités sont absents ou peu représentés ?
- quelle part des événements peut réellement être rapprochée de mesures ?
- qu'est-ce qu'une mesure horodatée permet de reconstituer indirectement sur l'activité humaine autour d'un équipement ?
- que conservez-vous, en brut ou en agrégat, et pourquoi ?

Toute conclusion tirée du parc instrumenté doit être accompagnée de son périmètre de validité.

In [ ]:
# TODO : comptages de couverture par site, type et criticité.
# TODO : part des événements rapprochables.

## 9. Décision

Concluez par l'un des trois statuts : `utilisable`, `utilisable sous conditions`, `non utilisable en l'état`.

Distinguez ce qui a été vérifié, ce qui a été transformé, ce qui reste incertain, les conditions à respecter avant M4 et le **coût de rejeu** de votre pipeline.

Les douze questions du brief trouvent leur réponse dans `docs/diagnostic_multisource.md`. Ce notebook porte les résultats qui les soutiennent.

In [ ]:
# TODO : synthèse chiffrée reprise dans la décision.